# FABLE Pyculator 2021 Scenario-Definition Patch

This notebook is a hands-on Phase 26 example for validating conservative edits to the 2021 FABLE-C `SCENARIOS definition` sheet.

No source workbook was mutated. A valid patch becomes generated-model input overrides keyed by workbook cell reference.

Expected local workbook artifact:

- `tmp/private-workbooks/2021_Open_FABLECalculator.xlsx`

Tracked example patch:

- `examples/scenario-definition-patches/fable_2021_diet_target_demo.yaml`

Optional generated-model artifact for the run cell:

- `tmp/generated-models/fable-2021/generated_fable_2021_model.py`

The generated model can be restored from:

- `examples/fable_2021/generated_fable_2021_model.py.xz`


In [2]:
from pathlib import Path
import lzma
import shutil

import pandas as pd
from IPython.display import Markdown, display

from fable_pyculator import (
    DEFAULT_2021_GENERATED_MODEL_PATH,
    DEFAULT_2021_WORKBOOK_PATH,
    build_2021_notebook_spec,
    editable_scenario_definition_cells,
    load_scenario_definition_patch,
    output_table_frame,
    run_2021_notebook_loop,
    validate_scenario_definition_patch,
)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "fable_pyculator").exists():
            return path
    raise RuntimeError("Could not find fable-pyculator repository root")


repo_root = find_repo_root(Path.cwd())
workbook_path = repo_root / DEFAULT_2021_WORKBOOK_PATH
generated_model_path = repo_root / DEFAULT_2021_GENERATED_MODEL_PATH
generated_model_archive_path = repo_root / "examples" / "fable_2021" / "generated_fable_2021_model.py.xz"
patch_path = repo_root / "examples" / "scenario-definition-patches" / "fable_2021_diet_target_demo.yaml"

if not generated_model_path.exists() and generated_model_archive_path.exists():
    generated_model_path.parent.mkdir(parents=True, exist_ok=True)
    with lzma.open(generated_model_archive_path, "rb") as source, generated_model_path.open("wb") as target:
        shutil.copyfileobj(source, target)

missing = [path for path in (workbook_path, patch_path) if not path.exists()]
ARTIFACTS_READY = not missing

display(Markdown(f"Repository root: `{repo_root}`"))
display(Markdown(f"Workbook path: `{workbook_path.relative_to(repo_root)}`"))
display(Markdown(f"Patch path: `{patch_path.relative_to(repo_root)}`"))
display(Markdown(f"Generated model path: `{generated_model_path.relative_to(repo_root)}`"))

if missing:
    display(Markdown("Missing required artifact(s):"))
    display([str(path.relative_to(repo_root)) for path in missing])
else:
    display(Markdown("Local workbook and patch are ready."))


Repository root: `/srv/shared-data/gafar001/fable-pyculator`

Workbook path: `tmp/private-workbooks/2021_Open_FABLECalculator.xlsx`

Patch path: `examples/scenario-definition-patches/fable_2021_diet_target_demo.yaml`

Generated model path: `tmp/generated-models/fable-2021/generated_fable_2021_model.py`

Local workbook and patch are ready.

## Discover Editable Cells

FABLE Pyculator derives the editable surface from workbook metadata. Phase 26 defaults to non-formula cells in columns tagged `DIRECT`.

In [3]:
if ARTIFACTS_READY:
    spec = build_2021_notebook_spec(workbook_path)
    editable_cells = editable_scenario_definition_cells(spec)
    display(Markdown(f"Discovered `{len(editable_cells):,}` editable scenario-definition cells."))
    display(pd.DataFrame([cell.to_dict() for cell in editable_cells[:10]]))
else:
    spec = None
    editable_cells = ()
    display(Markdown("Restore the 2021 workbook before discovering editable cells."))


Discovered `16,620` editable scenario-definition cells.

,table_name,table_label,sheet,cell_ref,row_label,column_label,column_role_tag,original_value,scenario_locations
0,scenarios_definition_dietimplrates,DietImplRates,SCENARIOS definition,SCENARIOS definition!X29,EATLancetAverage,DietScen,DIRECT,EATLancetAverage,[S.3.B]
1,scenarios_definition_dietimplrates,DietImplRates,SCENARIOS definition,SCENARIOS definition!Y29,EATLancetAverage,ImplTiming,DIRECT,Delayed,[S.3.B]
2,scenarios_definition_dietimplrates,DietImplRates,SCENARIOS definition,SCENARIOS definition!X30,FatDiet,DietScen,DIRECT,FatDiet,[S.3.B]
3,scenarios_definition_dietimplrates,DietImplRates,SCENARIOS definition,SCENARIOS definition!Y30,FatDiet,ImplTiming,DIRECT,Linear,[S.3.B]
4,scenarios_definition_dietimplrates,DietImplRates,SCENARIOS definition,SCENARIOS definition!X31,NatHealthyDiet,DietScen,DIRECT,NatHealthyDiet,[S.3.B]
5,scenarios_definition_dietimplrates,DietImplRates,SCENARIOS definition,SCENARIOS definition!Y31,NatHealthyDiet,ImplTiming,DIRECT,Linear,[S.3.B]
6,scenarios_definition_diettarget,DietTarget,SCENARIOS definition,SCENARIOS definition!AA29,EATLancetAverage,DietScen,DIRECT,EATLancetAverage,[S.3.C]
7,scenarios_definition_diettarget,DietTarget,SCENARIOS definition,SCENARIOS definition!AB29,EATLancetAverage,PROD_GROUP,DIRECT,CEREALS,[S.3.C]
8,scenarios_definition_diettarget,DietTarget,SCENARIOS definition,SCENARIOS definition!AA30,EATLancetAverage,DietScen,DIRECT,EATLancetAverage,[S.3.C]
9,scenarios_definition_diettarget,DietTarget,SCENARIOS definition,SCENARIOS definition!AB30,EATLancetAverage,PROD_GROUP,DIRECT,FISH,[S.3.C]


## Validate the Patch

The example patch uses a unique table/row/column selector. Validation resolves that selector to a workbook cell and produces generated-model input overrides. The workbook file is not edited.

In [4]:
patch = load_scenario_definition_patch(patch_path)
display(patch.to_dict())

if spec is not None:
    patch_result = validate_scenario_definition_patch(spec, patch)
    display(Markdown(f"Validated `{len(patch_result.edits)}` edit(s)."))
    display(pd.DataFrame([
        {
            "cell_ref": edit["cell"]["cell_ref"],
            "table": edit["cell"]["table_label"],
            "row_label": edit["cell"]["row_label"],
            "column_label": edit["cell"]["column_label"],
            "original_value": edit["cell"]["original_value"],
            "patched_value": edit["value"],
            "role_tag": edit["cell"]["column_role_tag"],
        }
        for edit in patch_result.edits
    ]))
    display(Markdown("Generated-model input overrides:"))
    display(patch_result.inputs)
    display(Markdown("No source workbook was mutated."))
else:
    patch_result = None
    display(Markdown("Skipped patch validation because the workbook spec is unavailable."))


{'version': 1,
 'patch_id': 'diet-target-demo',
 'edits': [{'value': 'Linear',
   'table': 'DietImplRates',
   'row_label': 'EATLancetAverage',
   'column_label': 'ImplTiming'}],
 'workbook_version': '2021',
 'description': 'Public-safe example patch showing the scenario-definition edit schema.'}

Validated `1` edit(s).

,cell_ref,table,row_label,column_label,original_value,patched_value,role_tag
0,SCENARIOS definition!Y29,DietImplRates,EATLancetAverage,ImplTiming,Delayed,Linear,DIRECT


Generated-model input overrides:

{'SCENARIOS definition!Y29': 'Linear'}

No source workbook was mutated.

## Optional Generated-Model Run

The run is gated. Review the validated patch first, then set `RUN_MODEL = True` if the local 2021 workbook and generated model are available.

In [5]:
RUN_MODEL = True

if RUN_MODEL:
    if not ARTIFACTS_READY:
        raise RuntimeError("Restore the 2021 workbook and patch before running the generated model.")
    if not generated_model_path.exists():
        raise RuntimeError(f"Generated model not found: {generated_model_path.relative_to(repo_root)}")

    result = run_2021_notebook_loop(
        {"gdp_scen": "SSP1"},
        workbook_path=workbook_path,
        generated_model_path=generated_model_path,
        scenario_definition_patch=patch,
        output_table_names=("ghg_resultsghg",),
        output_table_column_flavour_tags="OUTPUT-*",
        headline_series_names=("ghg_total_co2e",),
        include_figures=False,
    )
    display(Markdown("Generated model ran with the scenario-definition patch."))
    display({key: result.run.inputs[key] for key in patch_result.inputs})
    display(output_table_frame(result.run, "ghg_resultsghg", column_flavour_tags="OUTPUT-8").head())
    display(result.headline_frames["ghg_total_co2e"].head())
else:
    display(Markdown("Model run not started. Set `RUN_MODEL = True` after reviewing the patch validation output."))


Generated model ran with the scenario-definition patch.

{'SCENARIOS definition!Y29': 'Linear'}

,Year,LiveN2O,LiveCH4,CropCO2,CropN2O,CropCH4,DeforCO2,OtherLandCO2,LandSeq,BiofuelSavings
row,,,,,,,,,,
2000,2000,54.010285,124.206053,39.366663,53.722517,159.634835,0.000000,0.000000,0.000000,-0.810558
2005,2005,55.545018,126.017603,41.446155,53.448830,161.436065,9.175123,2.260630,-95.068094,-1.129594
2010,2010,62.635176,141.549979,42.028385,59.297512,174.631705,92.819474,115.655618,-118.415882,-1.366130
2015,2015,67.044633,150.837000,41.828791,58.530280,165.544881,60.887768,67.550145,-127.754997,-1.702214
2020,2020,70.424570,157.867297,41.687480,58.549958,162.085612,71.885766,92.950854,-164.177546,-2.096614


,name,label,group,year,value,unit,cell_refs,description
0,ghg_total_co2e,Total GHG emissions,GHG,2000,None,None,"(GHG!L18,)",None
1,ghg_total_co2e,Total GHG emissions,GHG,2005,None,None,"(GHG!L19,)",None
2,ghg_total_co2e,Total GHG emissions,GHG,2010,None,None,"(GHG!L20,)",None
3,ghg_total_co2e,Total GHG emissions,GHG,2015,None,None,"(GHG!L21,)",None
4,ghg_total_co2e,Total GHG emissions,GHG,2020,None,None,"(GHG!L22,)",None


In [6]:
if ARTIFACTS_READY:
    display(output_table_frame(result.run, "food_results_diets", column_flavour_tags='OUTPUT-*').head())
else:
    print("Skipped output inspection because required local artifacts are missing.")

,PROD_GROUP,YEAR,gr_trend,kcal_trend,fat_trend,prot_trend,gr_calc,kcal_calc,fat_calc,prot_calc,gr_healthy,kcal_healthy
row,,,,,,,,,,,,
CEREALS,CEREALS,2000,389.703716,1397.825305,4.243112,28.261134,389.703716,1397.825305,4.243112,28.261134,232.0,808.0
CEREALS,CEREALS,2005,397.850616,1427.614866,4.343846,28.820913,397.850616,1427.614866,4.343846,28.820913,232.0,808.0
CEREALS,CEREALS,2010,399.632824,1434.034243,4.385593,28.836216,399.632824,1434.034243,4.385593,28.836216,232.0,808.0
CEREALS,CEREALS,2015,373.140884,1338.971107,4.094869,26.924643,373.140884,1338.971107,4.094869,26.924643,232.0,808.0
CEREALS,CEREALS,2020,346.648944,1243.907971,3.804145,25.013070,346.648944,1243.907971,3.804145,25.013070,232.0,808.0


In [7]:
if ARTIFACTS_READY:
    display(output_table_frame(result.run, "food_total_results_diets").head())
else:
    print("Skipped output inspection because required local artifacts are missing.")

,PROD_GROUP,YEAR,kcal_hist,prot_hist,fat_hist,kcal_target,fat_target,prot_target,kcal_feas,fat_feas,prot_feas,kcal_MDER,prot_rec_low,prot_rec_up,fat_rec_low,fat_rec_up
row,,,,,,,,,,,,,,,,
TOTAL,TOTAL,2000,2316.126656,59.028987,49.970705,2316.126656,49.970705,59.028987,2314.243048,49.860197,58.852619,2035.010664,50.875267,178.063433,45.222459,67.833689
TOTAL,TOTAL,2005,2409.270566,61.491131,51.889467,2409.270566,51.889467,61.491131,2407.109819,51.762715,61.290948,2049.008982,51.225225,179.288286,45.533533,68.300299
TOTAL,TOTAL,2010,2497.289892,64.825039,56.349782,2497.289892,56.349782,64.825039,2495.085028,56.215974,64.618448,2056.984202,51.424605,179.986118,45.710760,68.566140
TOTAL,TOTAL,2015,NaN,NaN,NaN,2474.355225,60.493984,65.671586,2471.376859,60.325850,65.409073,2060.425857,51.510646,180.287262,45.787241,68.680862
TOTAL,TOTAL,2020,NaN,NaN,NaN,2451.420558,64.638186,66.518133,2448.381581,64.465976,66.246801,2063.640327,51.591008,180.568529,45.858674,68.788011


## Useful Feedback to Send Back

If something fails, the most useful details are:

- the patch YAML you used;
- the validation cell output;
- whether the target cell was missing, read-only, formula-valued, duplicated, or ambiguous;
- the generated-model run cell output, if you enabled `RUN_MODEL`.
